<a href="https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Model vs Baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring. Building on the label from ML-04 (`is_declining`) and the rule-based baseline from ML-05 (`ctr_below_tier_expectation`).

**Working month:** `2026-03`, split at `2026-03-16` — features from the first half, label from the second half. Same setup as ML-04, kept identical on purpose so this comparison is honest.


In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH_START = "2026-03-01"
MID_MONTH   = "2026-03-16"
MONTH_END   = "2026-04-01"

print("Connected. First half (features):", MONTH_START, "to", MID_MONTH)
print("Second half (label):", MID_MONTH, "to", MONTH_END)


Connected. First half (features): 2026-03-01 to 2026-03-16
Second half (label): 2026-03-16 to 2026-04-01


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Chosen: Random Forest classifier.** From Week 1's own result on the starter data, no single signal predicted decline well on its own (all correlations under 0.10 in ML-05's signal checks), but a Random Forest picked up the combination and beat a hand-written rule 3x on Precision@50. My lane needs a **probability score per page**, not just a label, because the deliverable is a ranked review queue — Random Forest gives me exactly that via `predict_proba`, the same way it did in Week 1.

I considered **Logistic Regression** (simpler, fully interpretable coefficients) and a single **Decision Tree** (fully readable, like the Week 1 example) as lighter alternatives. Both stay on the table as sanity checks, but Random Forest is my primary model because it captures feature interactions the other two structurally can't, and interaction is exactly what my ML-05 signal checks suggested was missing from a single-rule approach.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client (`client_hash_id`), 75/25 split.** Pages from the same client likely share templates, topics, and quality patterns. A plain random split could put page A of Client X in training and page B of Client X in testing — the model could then partly "recognize" that client's style rather than learning general signals about decline, inflating the test score. Holding out whole clients means the test set contains clients the model has never seen a single page from, which is the honest test for whether this generalizes.


In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Features: first half of March only (Mar 1-15) — same five signals as ML-04/ML-05
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions)                                    AS avg_impressions_h1,
        AVG(gsc_clicks)                                         AS avg_clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)       AS ctr_h1,
        AVG(gsc_avg_position)                                   AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MID_MONTH}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 500
""").df()

# Label: second half of March, same 20%-drop definition and volume floor fixed in ML-04
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MID_MONTH}' AND report_date < DATE '{MONTH_END}'
    GROUP BY 1, 2
""").df()

data = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_h2"] < 0.8 * data["avg_impressions_h1"] * 15).astype(int)
data = data.dropna()

print(f"{len(data):,} pages, {data['client_hash_id'].nunique()} clients, "
      f"declining rate {data['is_declining'].mean()*100:.1f}%")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Train: {len(train):,} pages, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} pages, {test['client_hash_id'].nunique()} clients")
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

41,816 pages, 33 clients, declining rate 28.1%
Train: 20,035 pages, 24 clients
Test:  21,781 pages, 9 clients
Client overlap between train and test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Metric: Precision@50** — same metric used throughout this track (Week 1, ML-05). Both rankings are built and scored on the exact same test set — the 25% of clients the model has never trained on.

**Baseline** (from ML-05): `ctr_gap × total_impressions` for pages in the `top_10`/`top_20` position tiers, using `expected_ctr` computed **from the training set only** (to keep this a fair comparison — no peeking at test-set CTR to build the baseline's own expectation). Pages outside those tiers get a baseline score of 0, exactly as ML-05 defined it.

**Model:** Random Forest, trained on the training split's five features, predicting `is_declining`. Test-set probabilities become the ranking.


In [4]:
from sklearn.ensemble import RandomForestClassifier

honest_features = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "days_with_impressions_h1"]

def position_tier(p):
    if p <= 3:  return "page_1_top3"
    if p <= 10: return "top_10"
    if p <= 20: return "top_20"
    return "beyond_20"

train["position_tier"] = train["avg_position_h1"].apply(position_tier)
test["position_tier"]  = test["avg_position_h1"].apply(position_tier)

# Baseline's expected CTR per tier, fit on TRAIN only
expected_ctr = train.groupby("position_tier")["ctr_h1"].mean().to_dict()
test["expected_ctr_tier"] = test["position_tier"].map(expected_ctr)

test["baseline_score"] = 0.0
eligible = test["position_tier"].isin(["top_10", "top_20"])
test.loc[eligible, "baseline_score"] = (
    (test.loc[eligible, "expected_ctr_tier"] - test.loc[eligible, "ctr_h1"]).clip(lower=0)
    * test.loc[eligible, "avg_impressions_h1"]
)

# Model
model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(train[honest_features], train["is_declining"])
test["model_score"] = model.predict_proba(test[honest_features])[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50
baseline_p50 = precision_at_k(test["baseline_score"], test["is_declining"], K)
model_p50    = precision_at_k(test["model_score"],    test["is_declining"], K)

comparison = pd.DataFrame({
    "method": ["baseline (ctr_below_tier_expectation)", "model (Random Forest)"],
    "precision_at_50": [round(baseline_p50, 3), round(model_p50, 3)],
})
comparison


,method,precision_at_50
0,baseline (ctr_below_tier_expectation),0.28
1,model (Random Forest),0.66


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


In [5]:
from sklearn.inspection import permutation_importance

# Feature importance, two ways
built_in_importance = pd.Series(model.feature_importances_, index=honest_features).sort_values(ascending=False)
print("Built-in feature importance:")
print(built_in_importance)
print()

perm = permutation_importance(model, test[honest_features], test["is_declining"], n_repeats=10, random_state=42, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=honest_features).sort_values(ascending=False)
print("Permutation importance (on held-out test set):")
print(perm_importance)


Built-in feature importance:
ctr_h1                      0.331172
days_with_impressions_h1    0.288807
avg_position_h1             0.134394
avg_clicks_h1               0.130127
avg_impressions_h1          0.115500
dtype: float64

Permutation importance (on held-out test set):
ctr_h1                      0.056724
avg_position_h1             0.005978
days_with_impressions_h1    0.004554
avg_clicks_h1              -0.012148
avg_impressions_h1         -0.019186
dtype: float64


In [6]:
# A close look at the model's top-20 queue: which ones were right, which were wrong?
test_sorted = test.sort_values("model_score", ascending=False).reset_index(drop=True)
top20 = test_sorted.head(20)

hits  = top20[top20["is_declining"] == 1]
misses = top20[top20["is_declining"] == 0]

print(f"Of the model's top 20: {len(hits)} were actually declining, {len(misses)} were not")
print()
print("A few of the model's false positives (flagged as declining, but weren't):")
print(misses[honest_features + ["model_score"]].head(5))


Of the model's top 20: 11 were actually declining, 9 were not

A few of the model's false positives (flagged as declining, but weren't):
    avg_impressions_h1  avg_clicks_h1    ctr_h1  avg_position_h1  \
1           475.066667       0.000000  0.000000        39.504589   
2           473.133333       0.000000  0.000000        38.932199   
5           795.800000       0.066667  0.000084        40.098914   
7           691.400000       0.000000  0.000000        35.461716   
11          545.066667       0.066667  0.000122        39.058148   

    days_with_impressions_h1  model_score  
1                         15     0.859737  
2                         15     0.859737  
5                         15     0.855777  
7                         15     0.854644  
11                        15     0.853893  


**Interpretation:** Both importance measures agree that `ctr_h1` is by far the strongest signal (built-in: 0.331, permutation: 0.057 — several times larger than any other feature in both rankings), followed by `days_with_impressions_h1` in the built-in ranking, though its permutation importance is much smaller (0.005) — suggesting the model leans on it less than the built-in measure implies once features are actually shuffled. `avg_clicks_h1` and `avg_impressions_h1` show slightly *negative* permutation importance, meaning shuffling them barely hurts (or even slightly helps) the model — they're largely redundant with `ctr_h1`, which already combines clicks and impressions into one ratio.

Looking at the model's top-20 queue: 11 of 20 were correctly declining (Precision@20 = 0.55, close to the reported Precision@50 = 0.66). The false positives share a clear pattern: very poor position (avg_position_h1 in the high-30s/40s) combined with near-zero CTR — pages that are already performing badly rather than pages that are actively declining. This suggests the model may be partly confusing "chronically weak" with "getting worse" — a real limitation, since my label only measures a further drop from an already-low base, and pages that bottomed out earlier have little room left to "decline" by the 20% threshold. A stronger label for the capstone would likely need to exclude already-bottomed-out pages from the target definition, or measure decline in absolute terms rather than relative percentage for very low-traffic pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] Model is compared to the ML-05 baseline on the same test set and the same metric (Precision@50)
- [ ] Split is grouped by client, with zero client overlap confirmed between train and test
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
